# Convert Imaris segmentation exports to LightSuite Sample Space v1

This notebook converts **Imaris** spot statistics and label-mask TIFF series into formats
accepted by `lightsuite brain import-annotations`:

| LightSuite format | Imaris source | Use when |
|-------------------|---------------|----------|
| `points_csv` | `*_Detailed.csv` (Statistics export) | Spot / cell positions (fast, small) |
| `mask_tiff` | `MASK/TIFF_Series_*/*_Z####.tif` | Full segmentation volume |

**Before you start:** run `lightsuite brain preprocess` on your sample so
`sample_reference.json` exists. Annotations must match that native grid — same shape and
voxel size as the stitched / acquired stack you segmented on (not the 20 µm registration preview).

See also: [Annotation import](../../docs/annotation_import.md) in the LightSuite docs.

## 0. Configure paths

Edit the variables below for your sample. Defaults point at the demo export under
`registration_imports/From Imaris`.

In [1]:
from pathlib import Path

# Where LightSuite wrote preprocess outputs (contains sample_reference.json)
SAVE_PATH = Path("/media/gbm/NVME2/ALICe-pipelines-data/JulieBuron/registered_allen")

IMARIS_EXPORT = Path(
    "/media/gbm/NVME2/ALICe-pipelines-data/registration_imports/From Imaris"
)
STATISTICS_CSV = IMARIS_EXPORT / "1-488-1x_Statistics/1-488-1x_Detailed.csv"
MASK_SLICE_DIR = IMARIS_EXPORT / "MASK/TIFF_Series_CH1"

OUTPUT_DIR = Path(
    "/media/gbm/NVME2/ALICe-pipelines-data/registration_imports/converted"
)
LABEL = "imaris_488"  # used in output filenames and YAML import.label

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("save_path:", SAVE_PATH)
print("imaris export:", IMARIS_EXPORT)
print("statistics:", STATISTICS_CSV)
print("mask slices:", MASK_SLICE_DIR)
print("output:", OUTPUT_DIR)

save_path: /media/gbm/NVME2/ALICe-pipelines-data/JulieBuron/registered_allen
imaris export: /media/gbm/NVME2/ALICe-pipelines-data/registration_imports/From Imaris
statistics: /media/gbm/NVME2/ALICe-pipelines-data/registration_imports/From Imaris/1-488-1x_Statistics/1-488-1x_Detailed.csv
mask slices: /media/gbm/NVME2/ALICe-pipelines-data/registration_imports/From Imaris/MASK/TIFF_Series_CH1
output: /media/gbm/NVME2/ALICe-pipelines-data/registration_imports/converted


## 1. Load the LightSuite native grid contract

After `preprocess`, LightSuite writes `sample_reference.json`. Every imported annotation must
align with this grid:

- **Shape** `(Y, X, Z)` = `shape_yxz` → `(ny, nx, nz)`
- **Voxel size** `[x, y, z]` in µm
- **Indices** are **1-based** — the first voxel center is `(1, 1, 1)`, not `(0, 0, 0)`
- **Axis order** for point CSVs is always `x, y, z`
- Do **not** apply `registration.orientation` yourself; import applies it during the warp

In [2]:
import json

ref_path = SAVE_PATH / "sample_reference.json"
if not ref_path.is_file():
    raise FileNotFoundError(
        f"Missing {ref_path}. Run: uv run lightsuite brain preprocess -c your_config.yaml"
    )

reference = json.loads(ref_path.read_text(encoding="utf-8"))
ny, nx, nz = reference["shape_yxz"]
voxel_um = reference["voxel_um"]

print(json.dumps(reference, indent=2))
print()
print(f"Expected mask shape (Y, X, Z): ({ny}, {nx}, {nz})")
print(f"Point bounds: 1 <= x <= {nx}, 1 <= y <= {ny}, 1 <= z <= {nz}")

{
  "format": "lightsuite_sample_space_v1",
  "sample_name": "Julie",
  "shape_yxz": [
    2048,
    2048,
    1361
  ],
  "voxel_um": [
    6.55,
    6.55,
    5.0
  ],
  "index_base": 1,
  "axis_order": "xyz",
  "coordinate_units": "voxel_indices",
  "orientation_applied": false
}

Expected mask shape (Y, X, Z): (2048, 2048, 1361)
Point bounds: 1 <= x <= 2048, 1 <= y <= 2048, 1 <= z <= 1361


## 2. Inspect the Imaris export

Typical layout from this demo:

```
From Imaris/
├── 1-488-1x_Statistics/
│   └── 1-488-1x_Detailed.csv    # Statistics → Export (Detailed)
└── MASK/
    └── TIFF_Series_CH1/
        ├── …_Z0000.tif            # one 2D label mask per Z (0-based index in filename)
        └── …_Z1360.tif
```

**Position file:** Imaris writes physical coordinates in **µm** (`Position X/Y/Z`, `Unit=µm`).
LightSuite expects **1-based native voxel indices** — convert with `int(pos_um / voxel_um) + 1`.

**Mask slices:** pixel values are Imaris **object IDs** (uint16 labels), not binary 0/255.
We binarize with `(plane > 0)` when stacking.

In [3]:
mask_files = sorted(MASK_SLICE_DIR.glob("*.tif*")) if MASK_SLICE_DIR.is_dir() else []
print("Statistics CSV:", STATISTICS_CSV.name if STATISTICS_CSV.is_file() else "MISSING")
print("Mask slice count:", len(mask_files))
if mask_files:
    print("First mask:", mask_files[0].name)
    print("Last mask:", mask_files[-1].name)

Statistics CSV: 1-488-1x_Detailed.csv
Mask slice count: 1361
First mask: TEMP DATA_Users_T00_C1_Z0000.tif
Last mask: TEMP DATA_Users_T00_C1_Z1360.tif


## 3. Read Imaris Statistics CSV

Imaris prepends a few header lines before the column names. We skip until the row starting with
`Position X` and parse comma-separated fields.

Expected columns: `Position X`, `Position Y`, `Position Z`, `Unit`, `Category`, `Collection`, …

In [4]:
import csv


def read_imaris_detailed_csv(path: Path) -> tuple[list[str], list[dict[str, str]]]:
    """Parse Imaris Statistics Detailed export (skip preamble lines)."""
    text = path.read_text(encoding="utf-8-sig")
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    header_idx = next(
        i for i, line in enumerate(lines) if line.startswith("Position X")
    )
    header = [col.strip() for col in lines[header_idx].split(",") if col.strip()]
    rows: list[dict[str, str]] = []
    for line in lines[header_idx + 1 :]:
        parts = [p.strip() for p in line.split(",")]
        if len(parts) < len(header):
            parts.extend([""] * (len(header) - len(parts)))
        rows.append(dict(zip(header, parts, strict=False)))
    return header, rows


if not STATISTICS_CSV.is_file():
    raise FileNotFoundError(f"Missing {STATISTICS_CSV}")

header, data_rows = read_imaris_detailed_csv(STATISTICS_CSV)
print(f"Loaded {STATISTICS_CSV.name}: {len(data_rows)} rows")
print("Columns:", header)
if data_rows:
    print("First row:", data_rows[0])

Loaded 1-488-1x_Detailed.csv: 236 rows
Columns: ['Position X', 'Position Y', 'Position Z', 'Unit', 'Category', 'Collection', 'Time', 'ID']
First row: {'Position X': '5118.87', 'Position Y': '9743.44', 'Position Z': '522.005', 'Unit': 'µm', 'Category': 'Spot', 'Collection': 'Position', 'Time': '1', 'ID': '884616'}


## 4. Convert spots → `points.csv`

LightSuite expects a CSV with header columns **`x`**, **`y`**, **`z`** (1-based native voxel indices).

Conversion from Imaris µm (image physical coordinates aligned with the segmented volume):

```python
x = int(Position_X_um / voxel_um[0]) + 1
y = int(Position_Y_um / voxel_um[1]) + 1
z = int(Position_Z_um / voxel_um[2]) + 1
```

If many points fall out of bounds, verify in Imaris that **Edit → Image Properties** matches the
native acquisition voxel size and that you segmented the full-resolution stack.

In [5]:
import numpy as np

POINTS_CSV = OUTPUT_DIR / f"{LABEL}_points.csv"


def imaris_um_to_lss_xyz(x_um: float, y_um: float, z_um: float, voxel_um: list[float]) -> tuple[float, float, float]:
    vx, vy, vz = (float(v) for v in voxel_um)
    return (
        int(x_um / vx) + 1.0,
        int(y_um / vy) + 1.0,
        int(z_um / vz) + 1.0,
    )


keep_category = "Spot"  # set to None to keep all rows
extra_cols = [c for c in ("ID",) if c in header]

points_xyz = []
with POINTS_CSV.open("w", newline="", encoding="utf-8") as fh:
    writer = csv.writer(fh)
    writer.writerow(["x", "y", "z"] + extra_cols)

    for row in data_rows:
        if keep_category is not None and row.get("Category") != keep_category:
            continue
        try:
            x_um = float(row["Position X"])
            y_um = float(row["Position Y"])
            z_um = float(row["Position Z"])
        except (KeyError, TypeError, ValueError):
            continue
        x, y, z = imaris_um_to_lss_xyz(x_um, y_um, z_um, voxel_um)
        extras = [row.get(c, "") for c in extra_cols]
        writer.writerow([x, y, z] + extras)
        points_xyz.append([x, y, z])

points_xyz = np.asarray(points_xyz, dtype=np.float64)
print(f"Wrote {len(points_xyz)} points → {POINTS_CSV}")
if len(points_xyz):
    print(
        "Ranges (1-based):",
        f"x [{points_xyz[:, 0].min():.1f}, {points_xyz[:, 0].max():.1f}]",
        f"y [{points_xyz[:, 1].min():.1f}, {points_xyz[:, 1].max():.1f}]",
        f"z [{points_xyz[:, 2].min():.1f}, {points_xyz[:, 2].max():.1f}]",
    )

Wrote 236 points → /media/gbm/NVME2/ALICe-pipelines-data/registration_imports/converted/imaris_488_points.csv
Ranges (1-based): x [490.0, 1518.0] y [104.0, 2032.0] z [68.0, 1269.0]


## 5. Validate points against `sample_reference.json`

Points outside `1 … nx/ny/nz` are dropped at import. Ideally everything should already be in range.

In [6]:
if len(points_xyz):
    in_bounds = (
        (points_xyz[:, 0] >= 1) & (points_xyz[:, 0] <= nx)
        & (points_xyz[:, 1] >= 1) & (points_xyz[:, 1] <= ny)
        & (points_xyz[:, 2] >= 1) & (points_xyz[:, 2] <= nz)
    )
    n_ok = int(in_bounds.sum())
    n_bad = len(points_xyz) - n_ok
    print(f"In bounds: {n_ok}/{len(points_xyz)} ({100 * n_ok / len(points_xyz):.1f}%)")
    if n_bad:
        bad = points_xyz[~in_bounds]
        print(f"WARNING: {n_bad} points out of bounds — will be dropped at import")
        print("First few bad points:", bad[:5])
else:
    print("No points to validate")

In bounds: 236/236 (100.0%)


## 6. Stack per-slice masks → single `mask.tif` (optional)

Skip this section if you only need spot positions.

Imaris exports one 2D **label** TIFF per Z plane. LightSuite expects one 3D **binary** TIFF:

- Shape `(Y, X, Z)` matching `shape_yxz`
- Values `0` / `255` (or `0` / `1`)
- Z-stack layout: one 2D page per Z slice

Filename `Z####` is **0-based**; page `i` corresponds to native **1-based** `z = i + 1`.

In [7]:
import re

import tifffile

BUILD_MASK = True  # set False to skip mask stacking
MASK_TIFF = OUTPUT_DIR / f"{LABEL}_mask.tif"

if not BUILD_MASK:
    print("Skipping mask build (BUILD_MASK=False)")
elif not mask_files:
    print("No mask slices found — skipping")
else:
    z_pattern = re.compile(r"_Z(\d+)$")

    def z_index(path: Path) -> int:
        match = z_pattern.search(path.stem)
        if match is None:
            raise ValueError(f"Cannot parse Z index from {path.name}")
        return int(match.group(1))

    sorted_masks = sorted(mask_files, key=z_index)
    z_indices = [z_index(p) for p in sorted_masks]
    print(f"Stacking {len(sorted_masks)} slices, Z index {z_indices[0]} … {z_indices[-1]}")

    if z_indices[-1] - z_indices[0] + 1 != len(sorted_masks):
        print("WARNING: Z indices may have gaps — check Imaris export completeness")

    planes = []
    for i, path in enumerate(sorted_masks):
        plane = tifffile.imread(path)
        if plane.ndim != 2:
            raise ValueError(f"Expected 2D mask slice in {path}, got shape {plane.shape}")
        planes.append((plane > 0).astype(np.uint8) * 255)
        if i % 200 == 0:
            print(f"  read {i}/{len(sorted_masks)} …")

    stack_zyx = np.stack(planes, axis=0)  # (Z, Y, X)
    print("Stack shape (Z, Y, X):", stack_zyx.shape)
    print("Nonzero voxels:", int(np.count_nonzero(stack_zyx)))

    yxz_shape = (stack_zyx.shape[1], stack_zyx.shape[2], stack_zyx.shape[0])
    if yxz_shape != (ny, nx, nz):
        print(
            f"WARNING: mask shape {yxz_shape} != reference ({ny}, {nx}, {nz}). "
            "Import will fail unless shapes match."
        )

    tifffile.imwrite(MASK_TIFF, stack_zyx, photometric="minisblack")
    print(f"Wrote mask → {MASK_TIFF}")

Stacking 1361 slices, Z index 0 … 1360
  read 0/1361 …
  read 200/1361 …
  read 400/1361 …
  read 600/1361 …
  read 800/1361 …
  read 1000/1361 …
  read 1200/1361 …
Stack shape (Z, Y, X): (1361, 2048, 2048)
Nonzero voxels: 31136133
Wrote mask → /media/gbm/NVME2/ALICe-pipelines-data/registration_imports/converted/imaris_488_mask.tif


## 7. Quick visual QC (optional)

Confirm points and/or mask alignment with the native acquisition grid before wiring
`import-annotations`.

Set **`QC_MODE`** in the next cell:

| Mode | Use when | Extra setup |
|------|----------|-------------|
| **`matplotlib`** | Low RAM, headless runs, quick static check | Default dev env (matplotlib) |
| **`napari`** | Interactive 3D slice browsing; GPU can speed rendering | `uv sync --extra gui` |

**Napari profiles** (`NAPARI_PROFILE`, only when `QC_MODE == "napari"`):

| Profile | RAM | What you see |
|---------|-----|--------------|
| **`light`** | Low | 20 µm registration preview + spots rescaled to that grid |
| **`native_lazy`** | Medium | Full-res mask (lazy/dask) + native spots |
| **`native_full`** | High | Full-res mask loaded entirely + native spots |

Pick **`matplotlib`** or **`napari` + `light`** on memory-constrained machines. Use
**`native_full`** only if you have enough RAM for the full native mask stack.

In [15]:
# --- Choose QC workflow ---
QC_MODE = "napari"  # "matplotlib" | "napari"

# Napari-only (ignored when QC_MODE == "matplotlib")
NAPARI_PROFILE = "light"  # "light" | "native_lazy" | "native_full"
NAPARI_REG_CHANNEL = 1  # chan_* used for the 20 µm preview background


def sample_xyz_to_napari_zyx(coords_xyz_1based: np.ndarray) -> np.ndarray:
    """Map 1-based sample (x, y, z) to Napari (z, y, x) indices."""
    pts = np.asarray(coords_xyz_1based, dtype=np.float64)
    if pts.size == 0:
        return np.zeros((0, 3), dtype=np.float64)
    return np.column_stack([pts[:, 2] - 1.0, pts[:, 1] - 1.0, pts[:, 0] - 1.0])


def native_xyz_to_preview_zyx(
    coords_xyz_1based: np.ndarray,
    *,
    native_shape_yxz: tuple[int, int, int],
    preview_shape_zyx: tuple[int, int, int],
) -> np.ndarray:
    """Rescale native 1-based xyz onto a downsampled preview grid for overlay."""
    ny_n, nx_n, nz_n = (int(v) for v in native_shape_yxz)
    pz, py, px = (int(v) for v in preview_shape_zyx)
    sx = (px - 1) / max(nx_n - 1, 1)
    sy = (py - 1) / max(ny_n - 1, 1)
    sz = (pz - 1) / max(nz_n - 1, 1)
    x = coords_xyz_1based[:, 0] - 1.0
    y = coords_xyz_1based[:, 1] - 1.0
    z = coords_xyz_1based[:, 2] - 1.0
    return np.column_stack([z * sz, y * sy, x * sx])


def _contrast_limits(volume: np.ndarray) -> tuple[float, float]:
    positive = volume[volume > 0]
    if positive.size:
        lo, hi = np.percentile(positive, (1.0, 99.5))
        return float(lo), float(hi)
    return float(volume.min()), float(volume.max())


def _load_tiff_zyx_for_napari(path: Path) -> np.ndarray:
    """Load a TIFF as Napari (Z, Y, X), stacking per-page IFDs when needed.

    ``tifffile.imread`` only returns the first plane for LightSuite registration
    preview TIFFs (one IFD per Z). Those must be stacked explicitly.
    """
    path = path.expanduser()
    data = np.asarray(tifffile.imread(path))
    if data.ndim == 3:
        return data
    if data.ndim != 2:
        msg = f"Unsupported TIFF shape {data.shape} in {path}"
        raise ValueError(msg)
    with tifffile.TiffFile(path) as tif:
        if len(tif.pages) <= 1:
            return data[np.newaxis, ...]
        planes = [np.asarray(page.asarray()) for page in tif.pages]
        return np.stack(planes, axis=0)


z_show = nz // 2
z0 = z_show - 1

if QC_MODE == "matplotlib":
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    if BUILD_MASK and MASK_TIFF.is_file():
        with tifffile.TiffFile(MASK_TIFF) as tif:
            mask_slice = tif.pages[z0].asarray()
        axes[0].imshow(mask_slice, cmap="gray")
        axes[0].set_title(f"Imaris mask slice z={z_show}")
    else:
        axes[0].set_title("No mask built")
        axes[0].axis("off")

    if len(points_xyz):
        sel = np.abs(points_xyz[:, 2] - z_show) <= 2.0
        sub = points_xyz[sel]
        axes[1].scatter(sub[:, 0], sub[:, 1], s=4, c="red", alpha=0.5)
        axes[1].set_xlim(1, nx)
        axes[1].set_ylim(ny, 1)
        axes[1].set_title(f"Points near z={z_show} ({int(sel.sum())} spots)")
    else:
        axes[1].set_title("No points")

    for ax in axes:
        ax.set_aspect("equal")
    plt.tight_layout()
    plt.show()

elif QC_MODE == "napari":
    try:
        import napari
    except ImportError as exc:
        raise ImportError(
            "Napari QC requires GUI extras: uv sync --extra gui"
        ) from exc

    viewer = napari.Viewer(title=f"LightSuite import QC — {LABEL}")
    napari_pts: np.ndarray | None = None

    if NAPARI_PROFILE == "light":
        regopts_path = SAVE_PATH / "regopts.json"
        if not regopts_path.is_file():
            raise FileNotFoundError(
                f"Missing {regopts_path}. Run preprocess first for reg-preview QC."
            )
        regopts = json.loads(regopts_path.read_text(encoding="utf-8"))
        registres_um = float(regopts["registres_um"])
        preview_path = SAVE_PATH / f"chan_{NAPARI_REG_CHANNEL}_sample_register_{int(registres_um)}um.tif"
        if not preview_path.is_file():
            raise FileNotFoundError(f"Missing registration preview: {preview_path}")

        preview_zyx = _load_tiff_zyx_for_napari(preview_path)
        clow, chigh = _contrast_limits(preview_zyx)
        viewer.add_image(
            preview_zyx,
            name=f"reg preview ch{NAPARI_REG_CHANNEL}",
            colormap="gray",
            contrast_limits=(clow, chigh),
            opacity=1.0,
        )
        if len(points_xyz):
            napari_pts = native_xyz_to_preview_zyx(
                points_xyz,
                native_shape_yxz=(ny, nx, nz),
                preview_shape_zyx=tuple(int(v) for v in preview_zyx.shape),
            )
        print(
            f"Napari light profile: {preview_path.name} "
            f"shape {tuple(int(v) for v in preview_zyx.shape)} (Z, Y, X)"
        )

    elif NAPARI_PROFILE in {"native_lazy", "native_full"}:
        if BUILD_MASK and MASK_TIFF.is_file():
            if NAPARI_PROFILE == "native_lazy":
                import dask.array as da

                with tifffile.TiffFile(MASK_TIFF) as tif:
                    mask_zyx = da.from_zarr(tif.aszarr())
            else:
                mask_zyx = _load_tiff_zyx_for_napari(MASK_TIFF)
            viewer.add_image(
                mask_zyx,
                name="segmentation mask",
                colormap="green",
                opacity=0.35,
                blending="additive",
            )
        elif BUILD_MASK:
            print("No mask file — showing points only")
        if len(points_xyz):
            napari_pts = sample_xyz_to_napari_zyx(points_xyz)

    else:
        raise ValueError(
            f"Unknown NAPARI_PROFILE={NAPARI_PROFILE!r}; "
            "use 'light', 'native_lazy', or 'native_full'"
        )

    if napari_pts is not None and napari_pts.shape[0]:
        viewer.add_points(
            napari_pts,
            name="spots",
            size=3,
            face_color="red",
            symbol="disc",
        )
        print(f"Showing {napari_pts.shape[0]} points")
    else:
        print("No points to display")

    napari.run()

else:
    raise ValueError(f"Unknown QC_MODE={QC_MODE!r}; use 'matplotlib' or 'napari'")

Napari light profile: chan_1_sample_register_20um.tif shape (1, 671, 671) (Z, Y, X)
Showing 236 points


## 8. YAML snippet for `import-annotations`

Add this block to your brain config YAML, then run **after** `lightsuite brain register`:

```bash
uv run lightsuite brain import-annotations -c examples/JulieBuron.yaml
uv run lightsuite brain inspect-imports -c examples/JulieBuron.yaml
```

Outputs land in `<save_path>/volume_registered/`.

In [9]:
import yaml

import_block = {
    "write_csv": True,
    "annotations": [
        {
            "format": "points_csv",
            "path": str(POINTS_CSV.resolve()),
            "label": f"{LABEL}_cells",
        },
    ],
}

if BUILD_MASK and MASK_TIFF.is_file():
    import_block["annotations"].append(
        {
            "format": "mask_tiff",
            "path": str(MASK_TIFF.resolve()),
            "label": f"{LABEL}_mask",
        }
    )

print("Copy into your config YAML:\n")
print(yaml.dump({"import": import_block}, default_flow_style=False, sort_keys=False))

Copy into your config YAML:

import:
  write_csv: true
  annotations:
  - format: points_csv
    path: /media/gbm/NVME2/ALICe-pipelines-data/registration_imports/converted/imaris_488_points.csv
    label: imaris_488_cells
  - format: mask_tiff
    path: /media/gbm/NVME2/ALICe-pipelines-data/registration_imports/converted/imaris_488_mask.tif
    label: imaris_488_mask



## 9. Test load with LightSuite adapters (optional)

Run from the LightSuite repo root with `uv sync`.

In [10]:
from lightsuite.config.models import AnnotationFormat, AnnotationImportConfig
from lightsuite.import_.adapters import load_annotation, load_points_csv, prepare_points_for_sample
from lightsuite.import_.sample_reference import load_sample_reference

ref = load_sample_reference(SAVE_PATH)

if POINTS_CSV.is_file():
    spec = AnnotationImportConfig(path=POINTS_CSV, format=AnnotationFormat.POINTS_CSV, label=LABEL)
    loaded = load_points_csv(spec)
    prepared = prepare_points_for_sample(loaded, reference=ref)
    print(f"Points: {loaded.coordinates.shape[0]} loaded → {prepared.coordinates.shape[0]} in bounds")

if BUILD_MASK and MASK_TIFF.is_file():
    spec = AnnotationImportConfig(path=MASK_TIFF, format=AnnotationFormat.MASK_TIFF, label=LABEL)
    mask = load_annotation(spec)
    print(f"Mask shape (Y,X,Z): {mask.volume.shape}, foreground voxels: {int(mask.volume.sum() > 0 and (mask.volume > 0).sum())}")

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
Points: 236 loaded → 236 in bounds


KeyboardInterrupt: 